In [ ]:
import subprocess, gc, os
import pandas as pd
import psutil

DATA_DIR = "/mnt/raid/data_sortedmob/RKD/RawFiles/encrypted"
FILE = "DTU_SJAELLAND_2020.tar.gz.aes-256-cbc"   # change if you want another year

def mem():
    vm = psutil.virtual_memory()
    me = psutil.Process(os.getpid()).memory_info().rss / 1e9
    print(f"Machine: {vm.percent:.0f}% used ({vm.available/1e9:.0f} GB free) | this kernel: {me:.1f} GB")

def open_stream(filename):
    """Decrypt + decompress into a pipe; nothing written to disk."""
    cmd = (f"openssl enc -d -aes-256-cbc -salt -pbkdf2 "
           f"-in {DATA_DIR}/{filename} -pass file:$HOME/.rkd_pass | tar xzO")
    return subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE)

mem()

In [ ]:
proc = open_stream(FILE)
peek = pd.read_csv(proc.stdout, encoding="ISO-8859-1", nrows=20, low_memory=False)
proc.kill(); proc.wait()

print(peek.dtypes)
peek.head(20)